# Data Modelling

In [1]:
import polars as pl
import duckdb

In [2]:
db_path = "/Users/kenim/Projects/premierlytics-data-platform/data/premierlytics.duckdb"
def query_db(query, db_path=db_path):
    with duckdb.connect(db_path, read_only=True) as con:
        result = con.query(query).pl()
    return result

## Matches Dataset

In [ ]:
query = "SELECT * FROM matches_bronze"

df_matches = query_db(query)

df_matches.head()

In [ ]:
df_matches.describe()

In [ ]:
df_matches['tournament'].describe()

## PlayerMatchStats

In [ ]:
query = "SELECT * FROM playermatchstats_bronze"

df_playermatchstats = query_db(query)

df_playermatchstats.head()

In [ ]:
df_playermatchstats.describe()

In [ ]:
df_playermatchstats.filter(pl.col('dispossessed').is_null()).describe()

## PlayerStats

In [ ]:
query = "SELECT * FROM playerstats_bronze"

df_playerstats = query_db(query)

df_playerstats.head()

In [ ]:
df_playerstats.describe()

In [ ]:
df_playerstats.filter(pl.col('first_name').is_null()).describe()

In [ ]:
print(df_playerstats.columns)

## Player Gameweek Stats

In [ ]:
query = "SELECT * FROM player_gameweek_stats_bronze"

df_playergameweekstats = query_db(query)

df_playergameweekstats.head()

In [ ]:
df_playergameweekstats.columns

## Fixtures

In [ ]:
query = "SELECT * FROM fixtures_bronze"

df_fixtures = query_db(query)

df_fixtures.head()

In [ ]:
df_fixtures.columns

In [ ]:
df_fixtures.describe()

In [ ]:
query = "SELECT COUNT(home_passes) FROM fixtures_bronze"


df_fixtures_psuedo = query_db(query)

df_fixtures_psuedo.head()

In [ ]:
# Check for missing matches
query = """
select count(*) as missing_count
from main_staging.stg_playermatchstats s
left join dim_match m on s.match_id = m.match_id
where m.match_sk is null;
"""
df_errors = query_db(query)
df_errors.head()

In [ ]:
query = """
select distinct s.match_id
from main_staging.stg_playermatchstats s
left join dim_match m on s.match_id = m.match_id
where m.match_sk is null
order by s.match_id;
"""
df_errors = query_db(query)
df_errors.head(20)

In [4]:
query = """
select
    s.id,
    s.gameweek,
    s.season,
    count(*) as match_count
from main_staging.stg_playerstats s
left join dim_player p
    on s.id = p.player_id
    and s.season = p.season
    and s.gameweek >= p.gameweek_effective_from
    and s.gameweek <= p.gameweek_effective_to
group by s.id, s.gameweek, s.season
having count(*) > 1
limit 10;
"""
df_errors = query_db(query)
df_errors

id,gameweek,season,match_count
i32,i32,str,i64
757,23,"""2024-2025""",24
137,23,"""2024-2025""",24
147,23,"""2024-2025""",24
618,23,"""2024-2025""",24
575,23,"""2024-2025""",24
696,23,"""2024-2025""",24
302,23,"""2024-2025""",24
754,23,"""2024-2025""",24
694,22,"""2024-2025""",25


In [5]:
query = """
select id, gameweek, season, ingested_at
from main_staging.stg_playerstats s
where id = 457 and gameweek = 23 and season = '2024-2025'
order by ingested_at;
"""
df_errors = query_db(query)
df_errors

id,gameweek,season,ingested_at
i32,i32,str,datetime[μs]
457,23,"""2024-2025""",2026-03-14 19:26:47.227829
457,23,"""2024-2025""",2026-03-14 19:26:48.739554
457,23,"""2024-2025""",2026-03-14 19:26:50.083987
457,23,"""2024-2025""",2026-03-14 19:26:51.429893
457,23,"""2024-2025""",2026-03-14 19:26:52.771764
…,…,…,…
457,23,"""2024-2025""",2026-03-14 19:27:15.418693
457,23,"""2024-2025""",2026-03-14 19:27:16.763778
457,23,"""2024-2025""",2026-03-14 19:27:18.751370
